In [ ]:
!pip install groq -q

import json
import time
from groq import Groq
from google.colab import userdata

In [ ]:
GROQ_API_KEY = "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"

In [ ]:
from google.colab import drive

drive.mount('/content/drive')

In [ ]:
import json
with open('/content/drive/MyDrive/masterthesis/30_arguments.json', 'r', encoding='utf-8') as f:
    arguments = json.load(f)
sample_item = arguments[0]
print(json.dumps(sample_item, indent=2))

In [ ]:
import json
import time
import os
from groq import Groq

client = Groq(api_key="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")

MODEL_ID = "meta-llama/llama-4-maverick-17b-128e-instruct"
NUM_RUNS = 10
OUTPUT_DIR = '/content/drive/MyDrive/masterthesis/llama-4-maverick/baseline_runs'
DATA_PATH = '/content/drive/MyDrive/masterthesis/30_arguments.json'


os.makedirs(OUTPUT_DIR, exist_ok=True)


try:
    with open(DATA_PATH, 'r', encoding='utf-8') as f:
        debate_data = json.load(f)
except FileNotFoundError:
    print("Error: Data file not found.")
    debate_data = []

if debate_data:
    print(f"Loaded {len(debate_data)} arguments. Starting {NUM_RUNS} runs...")


    for run_i in range(NUM_RUNS):
        print(f"\n>>> Starting Run {run_i+1}/{NUM_RUNS}...")

        current_run_results = []


        for item in debate_data:
            item_index = item.get("index")
            claim = item.get("claim", "")

            user_prompt = f"""
Consider the following claim:
"Claim: {claim}"

Your task:
Step 1: Decide whether you support or oppose the claim (write only "For" or "Against").
Step 2: Provide your reasoning in 3–4 sentences based on general, neutral reasoning.
Do NOT assume any persona, ideology, political theory, or worldview.

Output Format:
Stance: <For/Against>
Argument: <3–4 neutral reasoning sentences>
"""

            try:
                start_time = time.time()

                chat_completion = client.chat.completions.create(
                    model=MODEL_ID,
                    messages=[
                        {"role": "system", "content": "Provide a neutral analysis. Do not reveal AI identity."},
                        {"role": "user", "content": user_prompt}
                    ],
                    max_tokens=1000,
                    temperature=0.7
                )

                content = chat_completion.choices[0].message.content.strip()
                duration = time.time() - start_time


                current_run_results.append({
                    "index": item_index,
                    "run_id": run_i,
                    "model": MODEL_ID,
                    "response": content,
                    "duration": duration
                })


                print(f"[Run {run_i+1}] Processed Item {item_index} (Time: {duration:.2f}s)")

            except Exception as e:
                print(f"[Run {run_i+1}] Error on Item {item_index}: {str(e)}")
                time.sleep(5)

            time.sleep(0.5)


        save_path = os.path.join(OUTPUT_DIR, f'run_{run_i}.json')
        with open(save_path, 'w', encoding='utf-8') as f:
            json.dump(current_run_results, f, ensure_ascii=False, indent=4)

        print(f"--> Saved: {save_path}")

    print("\nAll 10 runs completed!")

else:
    print("No data found.")

In [ ]:
import pandas as pd
import json
import os
import re
import numpy as np


OUTPUT_DIR = '/content/drive/MyDrive/masterthesis/llama-4-maverick/baseline_runs'
NUM_FILES = 10


def extract_stance_robust(text):
    if not isinstance(text, str): return None

    match = re.search(r'Stance:\s*(For|Against)', text, re.IGNORECASE)
    if match:
        return match.group(1).lower()

    text_clean = text.strip().lower()
    if text_clean.startswith('for'): return 'for'
    if text_clean.startswith('against'): return 'against'
    return None


all_data = []

for i in range(NUM_FILES):
    file_path = os.path.join(OUTPUT_DIR, f'run_{i}.json')
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
            for item in data:
                item['run_id'] = i
                item['stance_clean'] = extract_stance_robust(item.get('response'))
            all_data.extend(data)
    else:
        print(f"Warning: {file_path} not found.")

df = pd.DataFrame(all_data)


df = df.dropna(subset=['stance_clean'])


flip_stats = []
grouped = df.groupby('index')

for name, group in grouped:
    counts = group['stance_clean'].value_counts()
    if len(counts) > 0:
        majority_count = counts.max()
        total_valid = counts.sum()


        flip_count = total_valid - majority_count
        flip_rate = flip_count / total_valid

        flip_stats.append({
            'index': name,
            'flip_rate': flip_rate,
            'total_valid': total_valid,
            'claim': group.iloc[0].get('claim', '')[:50] + "..." # 截取一部分claim方便看
        })

stats_df = pd.DataFrame(flip_stats)

if not stats_df.empty:
    global_mean = stats_df['flip_rate'].mean()
    global_std = stats_df['flip_rate'].std()

    print("="*60)
    print(f"Llama-4-Maverick Baseline Analysis (N={NUM_FILES})")
    print("-" * 60)
    print(f"Mean Flip Rate : {global_mean:.2%}")
    print(f"Standard Deviation :  {global_std:.2%}")
    print("="*60)


    print("\nTop 5 Most Unstable Questions:")
    print(stats_df.sort_values(by='flip_rate', ascending=False)[['index', 'flip_rate', 'claim']].head(5))
else:
    print("No valid statistics could be calculated.")